# Create Config Version (full feature list)

Builds `config/{model}/{dst_version}` from `{src_version}` so every feature enabled in `master_feature_encoding.csv` reaches the model.
- Source load list rows kept unchanged (target, exposure, fold, ...)
- Loss/exposure/premium/claim columns set to skip, never loaded as features
- Text columns stored as numbers added to `type_conversions.csv`
- Best params copied from source (05b off)
- Runner created at `runners/{dst_version}/run_{model}_{dst_version}.ipynb`

In [ ]:
src_version = "v1"
dst_version = "v3"
models = None        # None = all config/*/{src_version}; or e.g. ["car_coll"]
overwrite = False    # True = rebuild existing dst folders

In [ ]:
import sys
from pathlib import Path

project_root = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / 'current.pc').exists())
sys.path.insert(0, str(project_root / 'lib'))
from utils import setup_notebook_environment, load_config
from version_builder import build_full_feature_version, check_full_feature_version, create_version_runner, _aux_columns

project_root = setup_notebook_environment()
if models is None:
    models = sorted(p.parent.parent.name for p in Path('config').glob(f'*_*/{src_version}/config.yaml'))
print(f"{len(models)} models: {models}")

In [ ]:
# Build configs + runners, then validate
for m in models:
    build_full_feature_version(project_root, m, src_version, dst_version, overwrite=overwrite)
    create_version_runner(project_root, m, src_version, dst_version)
    aux = _aux_columns(load_config(f'config/{m}/{src_version}/config.yaml'), Path(f'config/{m}/{src_version}'))
    problems = check_full_feature_version(project_root, m, src_version, dst_version, aux_columns=aux)
    print(f"   check: {problems or 'OK'}")